### RAG with PDF 📄 Data extraction to give context to LLM 🧠

In [ ]:
%pip install pypdf

In [40]:
from pprint import pprint
from dotenv import load_dotenv
load = load_dotenv('../.env')

In [41]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
  base_url="http://localhost:11434",
  model="qwen3:latest",
  temperature=0.5,
  max_tokens=250,
)


### 1. Extracting the PDF files

In [42]:
from langchain_community.document_loaders import PyPDFLoader

pdf1 = "./attention.pdf"
pdf2 = "./LLMForgetting.pdf"
pdf3 = "./TestingAndEvaluatingLLM.pdf"

pdfFiles = [pdf1, pdf2, pdf3]

documents = []

for pdf in pdfFiles:
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

print(len(documents))

253


### 2. Text Splitting

In [43]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=1000, 
  chunk_overlap=200,
  add_start_index=True
)

all_splits = text_splitter.split_documents(documents)
len(all_splits)


640

### 3. Embedding

In [44]:
from langchain_ollama import OllamaEmbeddings

# Initialize the OpenAIEmbeddings object
# Use an embedding model, NOT a chat model like llama3.2
# Disable check_embedding_ctx_length to avoid OpenAI tokenizer issues with local models
embeddings = OllamaEmbeddings(model="llama3.2:latest")

vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

print(len(vector_1))
print(len(vector_2))
assert len(vector_1) == len(vector_2)

3072
3072


### 4. Vector Stores

In [ ]:
#%pip install -qU "langchain-chroma>=0.1.2"


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain_chroma import Chroma

# vector_store = Chroma.from_documents(
#   documents = all_splits,
#   embedding = embeddings,
#   persist_directory = "./chroma_langchain_db"
# )

### 5. Retrieving from the Persistent Vector Database

In [50]:
vector_store = Chroma(persist_directory="./chroma_langchain_db", embedding_function=embeddings)
result = vector_store.similarity_search("Types of LLM testing", k=3)
for doc in result:
    print(doc.page_content)

60 CHAPTER 4. LOGICAL REASONING CORRECTNESS
the following challenges: 1) If an LLM concludes correctly, it is unclear
whether the response stems from reasoning or merely relies on simple
heuristics such as memorization or word correlations (e.g., “dry floor”
is more likely to correlate with “playing football”). 2) If an LLM
fails to reason correctly, it is not clear which part of the reasoning
process it failed (i.e., inferring not raining from floor being dry or
inferring playing football from not raining). 3) There is a lack of
a system that can organize such test cases to cover all other formal
reasoning scenarios besides implication, such as logical equivalence
(e.g., If A then B, if B then A; therefore, A if and only if B). 4)
Furthermore, understanding an LLM’s performance on such test cases
provides little guidance on improving the reasoning ability of the
LLM. To better handle these challenges, a well-performing testing
26 CHAPTER 2. BACKGROUND REVIEW
Natural Language Generatio

In [54]:
result = vector_store.similarity_search_with_score("What are the types of LLM testing", k=3)
result[0]

(Document(id='c971b3f2-4694-4ac9-862c-2863ae88056d', metadata={'moddate': '2024-09-04T00:37:21+00:00', 'trapped': '/False', 'title': '', 'keywords': '', 'start_index': 0, 'creationdate': '2024-09-04T00:37:21+00:00', 'subject': '', 'creator': 'LaTeX with hyperref', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'author': '', 'source': './TestingAndEvaluatingLLM.pdf', 'page': 78, 'producer': 'pdfTeX-1.40.25', 'page_label': '60', 'total_pages': 223}, page_content='60 CHAPTER 4. LOGICAL REASONING CORRECTNESS\nthe following challenges: 1) If an LLM concludes correctly, it is unclear\nwhether the response stems from reasoning or merely relies on simple\nheuristics such as memorization or word correlations (e.g., “dry floor”\nis more likely to correlate with “playing football”). 2) If an LLM\nfails to reason correctly, it is not clear which part of the reasoning\nprocess it failed (i.e., inferring not raining from floor being dry o